# Notebook 6: Embeddings & Vectorstores – Turning Text into Searchable Vectors

**🎯 Goal:** Learn how to convert text into numerical representations (embeddings) and store them in a specialized database (a vectorstore) to enable efficient similarity searching.

## 🧩 Concept Overview

How can a computer understand that "king" is closer to "queen" than to "apple"? The answer is **embeddings**.

- **Embeddings:** These are numerical representations (vectors) of text. An embedding model is a deep learning model that reads text and outputs a list of numbers. Text with similar meaning will have similar vectors.
- **Vectorstores:** These are special databases designed to store and search these vectors very quickly. Think of it as a super-powered search engine for concepts, not just keywords.

The analogy is a library. Instead of organizing books by title (keyword search), you organize them by topic on shelves (similarity search). A book about "royal history" would be on the same shelf as "monarchies," even if the word "monarchy" isn't in the title.

## 🖼️ Visual Diagram

The process of creating and using a vectorstore:

```
Text Chunks from     +-------------------+      Numerical Vectors      +----------------+
Notebook 5      ===> | Embedding Model   | ===> [0.1, 0.9, ...]   ===> |                |
                     | (e.g., OpenAI)    |      [0.2, 0.8, ...]        |   Vectorstore  |
                     +-------------------+                             |   (e.g., FAISS)| 
                                                                       |                |
Query: "CEO speech"                                                    +-------+--------+
      |                                                                        |
      v                                                                        v
+-------------------+      Query Vector         +------------------------------------+
| Embedding Model   | ===> [0.11, 0.89, ...] => | Similarity Search -> Relevant Chunks |
+-------------------+                           +------------------------------------+
```

## ⚙️ Setup

We'll need `faiss-cpu` for the FAISS vectorstore, `chromadb` for a persistent store, and `sentence-transformers` for open-source embeddings. We'll also reuse our text splitting logic from the last notebook.

In [ ]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS, Chroma
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

load_dotenv()

# First, let's create our document chunks again for context
loader = TextLoader('../data/annual-report.pdf')
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(documents)

## 1. Embedding Models

LangChain supports many embedding providers. We'll look at two popular ones:
- **OpenAI:** High-quality, proprietary models.
- **Hugging Face:** Access to thousands of open-source models. We'll use a popular `sentence-transformers` model.

In [ ]:
# 🤖 Initialize OpenAI Embeddings
openai_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 🤗 Initialize Hugging Face Embeddings (runs locally on your CPU)
hf_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Let's test one by embedding a single piece of text
test_vector = openai_embeddings.embed_query("This is a test sentence.")
print(f"Vector dimension (OpenAI): {len(test_vector)}")

test_vector_hf = hf_embeddings.embed_query("This is a test sentence.")
print(f"Vector dimension (Hugging Face): {len(test_vector_hf)}")

## 2. Vectorstores

Now we'll store our embedded chunks in a vectorstore. 

### FAISS: Fast and In-Memory
`FAISS` (Facebook AI Similarity Search) is excellent for quick experiments. It's incredibly fast but runs in memory, meaning it's gone when your script ends.

In [ ]:
# 💾 Create a FAISS vectorstore from our document chunks and OpenAI embeddings
# This single line does a lot: downloads, chunks, embeds, and stores the data.
faiss_db = FAISS.from_documents(chunks, openai_embeddings)

print(f"FAISS database created. Number of vectors: {faiss_db.index.ntotal}")

### Chroma: Persistent and Feature-Rich
`Chroma` is a great choice for projects where you want the database to be saved to disk and loaded again later.

In [ ]:
# 💾 Create a Chroma vectorstore and persist it to a directory
chroma_db = Chroma.from_documents(
    chunks, 
    openai_embeddings, 
    persist_directory="../outputs/chroma_db" # The directory to save the database
)

print("Chroma database created and persisted.")

# You can load it back later like this:
# loaded_db = Chroma(persist_directory="./chroma_db", embedding_function=openai_embeddings)

## 3. Performing a Similarity Search

Now for the fun part! Let's ask a question and find the most relevant chunks from our annual report.

In [ ]:
query = "What were the key financial highlights of 2024?"

# 🔍 Search the FAISS database
# k=3 means 'return the top 3 most similar results'
results = faiss_db.similarity_search(query, k=3)

print(f"--- Search Results for: '{query}' ---")
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(f"Source: {doc.metadata.get('source', 'N/A')}")
    print(f"Content: \n{doc.page_content}")

## 🔧 Hands-On Exercise

**Goal:** Find movies with similar plots using the `movie-plots.csv` data.

1.  Load `../data/movie-plots.csv` using `CSVLoader`.
2.  Split the documents into chunks (size 300, overlap 0).
3.  Use the `HuggingFaceEmbeddings` model to create embeddings.
4.  Store the chunks in a `FAISS` vectorstore.
5.  Perform a similarity search for the query: `"a hero fights monsters in space"`.

In [ ]:
# --- Your Code Here ---

# 1. Load
movie_loader = CSVLoader('../data/movie-plots.csv', source_column='Title')
movie_docs = movie_loader.load()

# 2. Split
movie_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=0)
movie_chunks = movie_splitter.split_documents(movie_docs)

# 3. Embed (using Hugging Face)
hf_embeddings_exercise = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 4. Store
movie_db = FAISS.from_documents(movie_chunks, hf_embeddings_exercise)

# 5. Search
movie_query = "a hero fights monsters in space"
movie_results = movie_db.similarity_search(movie_query, k=3)

print(f"--- Movies similar to '{movie_query}' ---")
for doc in movie_results:
    print(f"- {doc.metadata['source']}")

## 🐞 Bug Bounty

A common mistake is to search for something that is semantically very different from the content of your documents. The vectorstore will still return the *closest* matches, but they won't be very relevant. This isn't a bug in the code, but a bug in the *process*.

**Task:** Search the `faiss_db` (our annual report) for `"what is the best recipe for lasagna?"`. Observe the results. They are the 'best' matches, but they are still completely irrelevant.

In [ ]:
irrelevant_query = "what is the best recipe for lasagna?"
irrelevant_results = faiss_db.similarity_search(irrelevant_query, k=1)

print(f"--- Search results for an irrelevant query ---")
print(irrelevant_results[0].page_content)
print("\nNotice how the result has nothing to do with lasagna. This is 'garbage in, garbage out'.")

## 💡 Pro Tip

The quality of your search results depends heavily on your **embedding model**. The `all-MiniLM-L6-v2` model is a great general-purpose starting point, but there are models trained specifically for different domains (e.g., finance, medicine, code). If you have specialized documents, searching for a domain-specific embedding model on the Hugging Face Hub can significantly improve your results.

## 🏁 Summary

You can now convert any text into a searchable knowledge base!

1.  **Embeddings turn text into numbers:** They capture the semantic meaning of text in a vector space.
2.  **Vectorstores are for searching vectors:** They are databases optimized for finding the most similar vectors to a given query vector.
3.  **The workflow is `Load -> Split -> Embed -> Store`:** This four-step process is the foundation of almost every Retrieval Augmented Generation (RAG) application.